# Free LLM Inference APIs: Groq & Google AI Studio

Two providers, one notebook. Each section covers:
1. A single (one-shot) API call
2. A multi-turn chat loop

**Prerequisites:**
```bash
pip install groq google-genai
```

Get your keys:
- Groq: https://console.groq.com/keys (no credit card)
- Gemini: https://aistudio.google.com/apikey (Google account)

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
%pip install -q groq google-genai

In [ ]:
# ── API Keys ───────────────────────────────────────────────────────────────────
# Option A: set environment variables before launching Jupyter
#   export GROQ_API_KEY="gsk_..."
#   export GEMINI_API_KEY="AIza..."
#
# Option B: paste here (don't commit to git!)
import os

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY",   "YOUR_GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")

---
## Part 1 — Groq

Groq's client is **OpenAI SDK-compatible**: same interface, just point to a different base URL.  
The native `groq` package wraps this automatically.

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

# ── 1a. Single inference call ──────────────────────────────────────────────────
GROQ_MODEL = "llama-3.3-70b-versatile"   # swap for any model from the markdown

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is the core idea behind Shapley values in one paragraph?"},
    ],
    temperature=0.7,
    max_tokens=300,
)

print("=== Groq single call ===")
print(response.choices[0].message.content)
print(f"\nUsage: {response.usage}")

In [ ]:
# ── 1b. Multi-turn chat with Groq ──────────────────────────────────────────────
# We maintain the full history list and append each turn manually.

def groq_chat(client, model: str, system: str = "You are a helpful assistant."):
    """Simple multi-turn chat loop. Type 'quit' to exit."""
    history = [{"role": "system", "content": system}]
    print(f"Chatting with {model} (type 'quit' to stop)\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        history.append({"role": "user", "content": user_input})

        resp = client.chat.completions.create(
            model=model,
            messages=history,
            temperature=0.7,
            max_tokens=512,
        )
        assistant_msg = resp.choices[0].message.content
        history.append({"role": "assistant", "content": assistant_msg})
        print(f"\nAssistant: {assistant_msg}\n")


# Uncomment to run interactively:
# groq_chat(groq_client, GROQ_MODEL, system="You are an expert in ML and explainability.")

In [ ]:
# ── 1b (scripted). Run a canned multi-turn exchange for demo/testing ───────────

def groq_scripted_chat(client, model: str, turns: list[str], system: str = "You are a concise assistant."):
    history = [{"role": "system", "content": system}]
    for user_msg in turns:
        history.append({"role": "user", "content": user_msg})
        resp = client.chat.completions.create(
            model=model, messages=history, temperature=0.7, max_tokens=300
        )
        assistant_msg = resp.choices[0].message.content
        history.append({"role": "assistant", "content": assistant_msg})
        print(f"User : {user_msg}")
        print(f"Model: {assistant_msg}\n")


groq_scripted_chat(
    groq_client,
    GROQ_MODEL,
    turns=[
        "Hi! What can you tell me about RLHF?",
        "How does the reward model relate to the policy in PPO?",
        "Give me a one-sentence summary of what we just discussed.",
    ],
)

---
## Part 2 — Google AI Studio (Gemini API)

Uses the `google-genai` SDK (not the older `google-generativeai`).  
The key difference from Groq: Gemini has a native `ChatSession` object that tracks history for you.

In [ ]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-2.5-flash"   # free-tier workhorse as of 2026

# ── 2a. Single inference call ──────────────────────────────────────────────────
response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="What is the core idea behind Shapley values in one paragraph?",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise assistant.",
        temperature=0.7,
        max_output_tokens=300,
    ),
)

print("=== Gemini single call ===")
print(response.text)
print(f"\nUsage: {response.usage_metadata}")

In [ ]:
# ── 2b. Multi-turn chat with Gemini ───────────────────────────────────────────
# The SDK's ChatSession object maintains history automatically.

def gemini_chat(client, model: str, system: str = "You are a helpful assistant."):
    """Interactive multi-turn chat loop. Type 'quit' to exit."""
    chat = client.chats.create(
        model=model,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0.7),
    )
    print(f"Chatting with {model} (type 'quit' to stop)\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        resp = chat.send_message(user_input)
        print(f"\nAssistant: {resp.text}\n")


# Uncomment to run interactively:
# gemini_chat(gemini_client, GEMINI_MODEL)

In [ ]:
# ── 2b (scripted). Canned multi-turn exchange ──────────────────────────────────

def gemini_scripted_chat(client, model: str, turns: list[str], system: str = "You are a concise assistant."):
    chat = client.chats.create(
        model=model,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0.7,
                                           max_output_tokens=300),
    )
    for user_msg in turns:
        resp = chat.send_message(user_msg)
        print(f"User : {user_msg}")
        print(f"Model: {resp.text}\n")


gemini_scripted_chat(
    gemini_client,
    GEMINI_MODEL,
    turns=[
        "Hi! What can you tell me about RLHF?",
        "How does the reward model relate to the policy in PPO?",
        "Give me a one-sentence summary of what we just discussed.",
    ],
)

---
## Key Differences at a Glance

| | Groq | Gemini (AI Studio) |
|---|---|---|
| **Client** | `Groq(api_key=...)` | `genai.Client(api_key=...)` |
| **Call style** | `client.chat.completions.create(messages=[...])` | `client.models.generate_content(contents=...)` |
| **Chat history** | Managed manually (append to `messages` list) | Managed by `ChatSession` object |
| **System prompt** | `{"role": "system", "content": ...}` in messages | `GenerateContentConfig(system_instruction=...)` |
| **Response text** | `response.choices[0].message.content` | `response.text` |
| **OpenAI compat** | ✅ Drop-in with `openai` SDK + custom base URL | ❌ Own SDK only |